In [7]:
# torchaudio uses TorchCodec for WAV decoding in recent releases
%pip install -q torchcodec


Note: you may need to restart the kernel to use updated packages.


In [3]:
# Core libraries
import os
import time
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

import torchaudio
from torchaudio.datasets import SPEECHCOMMANDS

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cpu")
print("Using device:", device)


Using device: cpu


Data Loading and Pre processing

In [4]:
# Choosing keyword subset 
# Full label set has 35 words; we pick a manageable subset (edit as you like)
SELECTED_LABELS = ["yes", "no", "up", "down", "left", "right",
                    "on", "off", "stop", "go"]
LABEL_TO_IDX = {label: i for i, label in enumerate(SELECTED_LABELS)}
NUM_CLASSES = len(SELECTED_LABELS)
print(f"Using {NUM_CLASSES} keywords: {SELECTED_LABELS}")


Using 10 keywords: ['yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go']


In [6]:
# Download dataset
DATA_ROOT = "./data"
os.makedirs(DATA_ROOT, exist_ok=True)

train_raw = SPEECHCOMMANDS(root=DATA_ROOT, download=True, subset="training")
val_raw   = SPEECHCOMMANDS(root=DATA_ROOT, download=True, subset="validation")
test_raw  = SPEECHCOMMANDS(root=DATA_ROOT, download=True, subset="testing")

print("Train / Val / Test raw sizes:", len(train_raw), len(val_raw), len(test_raw))


Train / Val / Test raw sizes: 84843 9981 11005


In [10]:
SAMPLE_RATE = 16000
N_MELS = 40
FIXED_LENGTH = SAMPLE_RATE  # pad/truncate every clip to exactly 1 second

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE, n_fft=400, hop_length=160, n_mels=N_MELS
)
db_transform = torchaudio.transforms.AmplitudeToDB()

def waveform_to_features(waveform):
    """Pad/truncate waveform to FIXED_LENGTH samples, then convert to log-Mel spectrogram."""
    if waveform.shape[1] < FIXED_LENGTH:
        pad = FIXED_LENGTH - waveform.shape[1]
        waveform = F.pad(waveform, (0, pad))
    else:
        waveform = waveform[:, :FIXED_LENGTH]
    mel = mel_transform(waveform)
    mel_db = db_transform(mel)
    return mel_db  # shape: [1, N_MELS, time_frames]


class KeywordSpottingDataset(Dataset):
    """Wraps a torchaudio SPEECHCOMMANDS split, keeping only SELECTED_LABELS."""

    def __init__(self, raw_dataset, label_to_idx):
        self.label_to_idx = label_to_idx
        self.items = [
            i for i in range(len(raw_dataset))
            if raw_dataset[i][2] in label_to_idx
        ]
        self.raw_dataset = raw_dataset

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        waveform, sample_rate, label, *_ = self.raw_dataset[self.items[idx]]
        features = waveform_to_features(waveform)
        target = self.label_to_idx[label]
        return features, target


train_ds = KeywordSpottingDataset(train_raw, LABEL_TO_IDX)
val_ds   = KeywordSpottingDataset(val_raw, LABEL_TO_IDX)
test_ds  = KeywordSpottingDataset(test_raw, LABEL_TO_IDX)

print("Filtered Train / Val / Test sizes:", len(train_ds), len(val_ds), len(test_ds))

# Peek at one sample's feature shape
sample_feat, sample_label = train_ds[0]
print("Feature shape (channels, n_mels, time_frames):", sample_feat.shape)


RuntimeError: Failed to create AudioDecoder for ./data\SpeechCommands\speech_commands_v0.02\backward\0165e0e8_nohash_0.wav: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, 8, and 9, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.14.1+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load 

In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
